# chain-rule-elementwise — ex3: leaky_relu_back — parameterized elementwise back fn

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `chain-rule-elementwise`. Running the final beacon cell reports progress against the `Backprop: Elementwise chain rule` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import torch.nn.functional as F

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: Elementwise chain rule` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`chain-rule-elementwise`** (exercise 3). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "chain-rule-elementwise"
DD_SUBTOPIC = "Backprop: Elementwise chain rule"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Parameterized elementwise: `leaky_relu_back` carries the slope through

Ex1 (sigmoid/relu) and ex2 (tanh/softplus) used `(grad_out, out, x)` — no extra parameters. The deepening move is the FIRST parameterized elementwise op: `leaky_relu(x, negative_slope=0.01)` has TWO regimes.

```python
# Forward: out = x if x > 0 else negative_slope * x
# Derivative: 1 where x > 0, negative_slope where x <= 0.
# Backward signature must accept the slope so the reverse pass can
# read it back from recipe.kwargs and pass it through.
def leaky_relu_back(grad_out, out, x, negative_slope=0.01):
    local = t.where(x > 0, t.ones_like(x), t.full_like(x, negative_slope))
    return grad_out * local
```

**Why the kwarg matters at backward time.** Inside the reverse pass, `leaky_relu_back(grad_out, out, x, **recipe.kwargs)` re-uses the same slope the forward used. Hard-coding `0.01` would silently produce the wrong gradient whenever the user changed it. This is the same kwargs-pass-through pattern from the recipe atoms, applied to a back fn.

**Slope = 0 reduces to ReLU.** When `negative_slope == 0`, `local` is `1` for positive x and `0` for non-positive x — exactly relu's gradient. This is a useful invariant to test: the deepening drill checks that the parameterized form REDUCES to the ex1 baseline.

### Exercise 3 — leaky_relu_back — parameterized elementwise back fn

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply the elementwise chain rule to a parameterized op by writing `leaky_relu_back(grad_out, out, x, negative_slope)` so the slope kwarg threads through and the derivative is `1` where `x > 0`, `negative_slope` elsewhere.
> Keywords: leaky-relu, chain-rule, kwarg, parameterized
> ```

**KCs targeted:** `chain-rule-elementwise`, `back-fn-accepts-forward-kwargs`

Implement `ex3_leaky_relu_back(grad_out, out, x, negative_slope=0.01)`.

Forward op (for context): `out = leaky_relu(x, negative_slope) = x if x > 0 else negative_slope * x` (elementwise).

Local derivative is piecewise:
- `1` where `x > 0`
- `negative_slope` where `x <= 0`

Return `grad_out * local_derivative` with shape == `x.shape`.

Constraints:

1. The `negative_slope` argument MUST be a keyword arg with default `0.01` (matching `torch.nn.functional.leaky_relu`).
2. Use `t.where(x > 0, ..., ...)` or a `(x > 0).float()` mask — either is fine, but the output dtype must match `grad_out.dtype`.
3. The function must work for any tensor dtype (float32, float64) and any shape (0-D scalar, 1-D, 2-D, etc.).
4. The `out` argument is provided for signature consistency with ex1/ex2 — you don't have to use it (leaky_relu's derivative depends on `x`, not `out`).

In [ ]:
def ex3_leaky_relu_back(grad_out, out, x, negative_slope=0.01):
    """Elementwise back fn for leaky_relu — slope threads through as kwarg."""
    raise NotImplementedError()


def _test_ex3():
    # === Default slope (0.01) — most common case ===
    x = t.tensor([-2.0, -0.5, 0.0, 0.5, 2.0])
    out = t.where(x > 0, x, 0.01 * x)
    grad_out = t.ones_like(x)
    g = ex3_leaky_relu_back(grad_out, out, x)
    expected = t.tensor([0.01, 0.01, 0.01, 1.0, 1.0])
    # Note: at x == 0, convention is to use the negative-side slope.
    assert g.shape == x.shape, f'shape mismatch: {g.shape}'
    assert t.allclose(g, expected), f'default slope: got {g}, expected {expected}'

    # === Custom slope (0.2 — a typical PReLU choice) ===
    g = ex3_leaky_relu_back(grad_out, out, x, negative_slope=0.2)
    expected = t.tensor([0.2, 0.2, 0.2, 1.0, 1.0])
    assert t.allclose(g, expected), f'slope=0.2: got {g}'

    # === Slope=0 reduces to relu_back ===
    g = ex3_leaky_relu_back(grad_out, out, x, negative_slope=0.0)
    expected_relu = (x > 0).to(grad_out.dtype)
    assert t.allclose(g, expected_relu), f'slope=0 must equal relu: got {g}'

    # === Slope=1 reduces to identity (linear pass-through) ===
    g = ex3_leaky_relu_back(grad_out, out, x, negative_slope=1.0)
    assert t.allclose(g, grad_out), f'slope=1 must equal grad_out: got {g}'

    # === Non-unit grad_out — chain-rule scaling ===
    grad_out2 = t.tensor([10.0, 20.0, 30.0, 40.0, 50.0])
    g = ex3_leaky_relu_back(grad_out2, out, x, negative_slope=0.01)
    expected = t.tensor([0.01*10, 0.01*20, 0.01*30, 1.0*40, 1.0*50])
    assert t.allclose(g, expected), f'scaled grad_out wrong: got {g}'

    # === 2-D shape ===
    x = t.randn(4, 5)
    out = t.where(x > 0, x, 0.01 * x)
    grad_out = t.randn(4, 5)
    g = ex3_leaky_relu_back(grad_out, out, x)
    assert g.shape == (4, 5)
    # Manually compute expected
    local = t.where(x > 0, t.ones_like(x), 0.01 * t.ones_like(x))
    assert t.allclose(g, grad_out * local), 'matrix-shape leaky_relu_back wrong'

    # === Cross-check vs torch.autograd ===
    x = t.randn(10, requires_grad=True)
    out_t = t.nn.functional.leaky_relu(x, negative_slope=0.05)
    out_t.sum().backward()
    ours = ex3_leaky_relu_back(t.ones_like(out_t), out_t.detach(), x.detach(), negative_slope=0.05)
    assert t.allclose(x.grad, ours, atol=1e-6), f'autograd mismatch: {x.grad} vs {ours}'

    # === Scalar (0-D) input ===
    x = t.tensor(-3.0)
    out = t.tensor(-0.03)
    g = ex3_leaky_relu_back(t.tensor(1.0), out, x, negative_slope=0.01)
    assert g.dim() == 0
    assert abs(g.item() - 0.01) < 1e-6, f'0-D negative case: {g.item()}'

    # === Float64 ===
    x = t.tensor([-1.0, 1.0], dtype=t.float64)
    out = t.where(x > 0, x, 0.01 * x)
    g = ex3_leaky_relu_back(t.ones_like(x), out, x)
    assert g.dtype == t.float64, f'dtype must match grad_out: got {g.dtype}'
    _dd_passed.add('ex3')
    print("ex3 ✓")

_test_ex3()

<details><summary>Solution</summary>

```python
def ex3_leaky_relu_back(grad_out, out, x, negative_slope=0.01):
    local = t.where(
        x > 0,
        t.ones_like(x),
        t.full_like(x, negative_slope),
    )
    return grad_out * local
```

**`t.where(cond, true_val, false_val)` over `*`.** Multiplying by `(x > 0).float() + negative_slope * (x <= 0).float()` works but allocates two masks. `t.where` does it in one pass.

**`t.full_like(x, negative_slope)` matches dtype/device of `x`.** Using a bare Python float on the false-branch would force PyTorch to cast, sometimes silently upgrading to float64 and breaking downstream dtype expectations.

**Convention at `x == 0`.** Mathematically the derivative is undefined exactly at the kink. PyTorch and most frameworks use the negative-side slope. Our `x > 0` (strict) condition matches that convention — at zero, we fall into the false branch.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex3',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()